In [ ]:
# @title Setup — Run this first
import os, sys

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    if not os.path.exists('/content/MS_spectra_encoding'):
        !git clone https://github.com/Alexander-Sol/MS_spectra_encoding.git /content/MS_spectra_encoding

    %cd /content/MS_spectra_encoding

    # Install packages not pre-installed on Colab
    !pip install -q pyteomics==4.6.1 spectrum-utils==0.4.2 rapidhash==0.1.0 mmh3

    # The DDA data file is force-committed, so it clones with the repo.
    # gdown fallback (uncomment + fill FILE_ID if the committed copy is unavailable):
    # !gdown <FILE_ID> -O Data/04-17-23_CA_Tryp_HCD_10min-calib.mzML
else:
    # Running locally: use this checkout as-is (pip install -r requirements.txt).
    print(f"Local run - using repo at {os.getcwd()}")

# 2. PSM Feature Vectors

## Goals:

In this notebook, we explore the earliest and simplest approach to encoding mass spectra for machine learning: a **PSM feature vector**.

1. [Understand what a PSM is and why we need a numerical representation.](#21-what-is-a-psm)
2. [See what a database search reports for each spectrum.](#22-database-search)
3. [Explore the hand-crafted features Percolator (2007) uses to represent each PSM, and the ones MetaMorpheus reports directly.](#23-the-percolator-feature-vector)
4. [Compute a real feature vector from the VLDALDSIK PSM.](#24-computing-the-features)
5. [Understand the limitations of this encoding and how later notebooks address them.](#25-limitations-and-what-comes-next)

In [1]:
# @title Run this cell to import all necessary packages
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import math
from util import *

## 2.1 What is a PSM?

In Notebook 1, we saw how a mass spectrum is produced by fragmenting a peptide and recording the masses of the resulting fragment ions. We also saw a **peptide-spectrum match (PSM)**: a candidate assignment of a peptide sequence to an observed spectrum, produced by a database search tool.

We claimed that **scan 5672** was matched to **VLDALDSIK** from bovine carbonic anhydrase. However, how do we know this? Is it possible that scan 5672 corresponds to a different peptide? Is it possible that the matches between the observed fragment ions and the theoretical fragment ions of **VLDALDSIK** occured solely by chance? When tools report a list of peptide-spectrum matches, they have to distinguish between the confident PSMs, where the match between theoretical and experimental fragment ions is very good, and low-quality PSMs, where the match between theoretical and experimental ions occured due to chance. 

To distinguish between good and bad PSMs, the most common approach is called the target-decoy strategy. In target-decoy, spectra are searched against real peptide sequences, corresponding to peptides from proteins that are expected to be in the sample, and decoy peptide sequences, peptides that don't correspond to any real proteins. Any PSMs where a spectrum is matched to a decoy peptide is a bad PSM, as the peptide doesn't really exist! Therefore, the match can only happen by chance.

Decoy peptides are commonly generated by taking the sequence of a real peptide, holding the terminal amino acids in place, and scrambling everything in-between. For example:

Target Peptide: **VLDALDSIK**
Decoy Peptide: **VDLSADLIK**


In [2]:
mzml_path = 'Data/04-17-23_CA_Tryp_HCD_10min-calib.mzML'
peptide   = 'VLDALDSIK'
scan      = 5672

spectrum = get_MS2_object(mzml_path, scan, peptide=peptide)
plot_MS2(spectrum, title=f'Scan {scan}: {peptide}')

In [5]:
mzml_path = 'Data/04-17-23_CA_Tryp_HCD_10min-calib.mzML'
peptide   = 'VDLSADLIK'
scan      = 6178

spectrum = get_MS2_object(mzml_path, scan, peptide=peptide)
plot_MS2(spectrum, title=f'Scan {scan}: {peptide}')

### How well can the *decoy* match?

We just looked at the target **VLDALDSIK**. Its decoy, **ISDLADLVK**, corresponds to no real protein, so any spectrum it appears to "match" is a coincidence. To see how large a coincidence to expect, let's run a mini database search with a database of exactly one peptide, the decoy, and scan the **entire file** for the MS2 spectrum it fits best.

We score each spectrum the way Notebook 1 described step 3 of database search: build the theoretical b/y ion ladder for the decoy and count how many of those ions actually appear in the observed spectrum.

In [ ]:
# Search the whole file for the MS2 spectrum that best matches the DECOY peptide.
# A decoy corresponds to no real protein, so its best match tells us how good a
# peptide-spectrum match we should expect purely by chance.
decoy = 'ISDLADLVK'   # VLDALDSIK reversed, keeping the C-terminal K

def theoretical_by_ions(peptide, max_charge=1):
    """Theoretical b/y fragment m/z for a peptide, at charges 1..max_charge.
    This is the same b/y ladder that make_ion_ladder() builds in Notebook 1."""
    ions = []
    b = 0.0                                   # running b-ion (N-terminal) neutral mass
    for aa in peptide[:-1]:
        b += AMINO_ACID_DICT[aa]
        ions += [(b + z * PROTON_MASS) / z for z in range(1, max_charge + 1)]
    y = HYDROGEN_MASS + OXYGEN_MASS           # running y-ion (C-terminal) neutral mass (+OH)
    for aa in peptide[::-1][:-1]:
        y += AMINO_ACID_DICT[aa]
        ions += [(y + z * PROTON_MASS) / z for z in range(1, max_charge + 1)]
    return np.array(sorted(ions))

def match_score(spectrum, theo_mz, tol_ppm=20):
    """(number of theoretical ions that land on an observed peak within tol_ppm,
        fraction of the spectrum's intensity those matched peaks account for)."""
    obs, inten = np.asarray(spectrum.mz), np.asarray(spectrum.intensity)
    if obs.size == 0:
        return 0, 0.0
    matched, matched_int = 0, 0.0
    for t in theo_mz:
        hit = np.abs(obs - t) <= t * tol_ppm / 1e6
        if hit.any():
            matched += 1
            matched_int += inten[hit].max()
    return matched, (matched_int / inten.sum() if inten.sum() else 0.0)

theo = theoretical_by_ions(decoy)

# get_all_MS2_objects reads every MS2 scan in the file, so this cell is slow the first time.
best_scan, best = None, (-1, -1.0)
for s in get_all_MS2_objects(mzml_path):
    score = match_score(s, theo)          # tuples compare lexicographically:
    if score > best:                      # rank by matched-ion count, then intensity
        best, best_scan = score, extract_scan_number(s.identifier)

print(f'Best match for decoy {decoy}: scan {best_scan} '
      f'-- {best[0]}/{len(theo)} b/y ions matched, '
      f'{best[1]:.1%} of the spectrum\'s intensity explained')

# Plot that best-matching spectrum, annotated with the decoy's b/y ions.
best_spectrum = get_MS2_object(mzml_path, best_scan, peptide=decoy)
plot_MS2(best_spectrum, title=f'Scan {best_scan}: best match for decoy {decoy}')

Compare this to the target: **VLDALDSIK** on scan 5672 matched **10 of its 16** b/y ions (we saw this spectrum above). The decoy's *best* hit anywhere in the ~3,000 spectra is much weaker, and it lands on a spectrum whose precursor m/z (~728, 2+) is nowhere near the decoy's own mass, so the overlap is pure coincidence: this spectrum could not actually have come from ISDLADLVK.

That gap between real matches and the best a decoy can do by chance is exactly what the target-decoy strategy exploits. Decoy PSMs map out the score distribution of random matches, which lets us set a threshold that separates confident identifications from noise.

### The whole file at once: targets vs. decoys

The single decoy above was one anecdote. In practice, a search engine competes a **target** database and a matched **decoy** database against *every* spectrum. We searched this file with [MetaMorpheus](https://github.com/smith-chem-wisc/MetaMorpheus), which writes one row per PSM to an `AllPSMs.psmtsv` file (copied here to `Data/MetaMorpheus_AllPSMs.psmtsv`). Each row has a `Score` column (roughly, the number of matched fragment ions plus a fractional bonus for their intensities) and a `Decoy/Contaminant/Target` label.

Below we load that file and keep only PSMs that are (1) **unmodified** (their `Full Sequence` carries no modifications, so it equals the `Base Sequence`), (2) unambiguous target/decoy calls, and (3) on a scan listed in `Data/ScanNumsWPsms.txt`, so we look at the same scans used elsewhere in this tutorial.

If the score is informative, targets should pile up at higher values than decoys.

We also add one column of our own: the **fraction of theoretical fragments observed**. A peptide of length $n$ has $n-1$ b ions and $n-1$ y ions, so we divide MetaMorpheus' `Matched Ion Counts` by $2(n-1)$.

In [ ]:
# MetaMorpheus search result for this file (one row per PSM).
mm_path = 'Data/MetaMorpheus_AllPSMs.psmtsv'
psms = pd.read_csv(mm_path, sep='\t', low_memory=False)

# Restrict to the scans that have a PSM listed in ScanNumsWPsms.txt.
psm_scans = set(pd.read_csv('Data/ScanNumsWPsms.txt')['Scan Number'].astype(int))

# Keep only UNMODIFIED peptides (no bracketed mods -> Full Sequence == Base Sequence),
# unambiguous target/decoy calls (drop 'T|D'-style ambiguous rows), and matching scans.
psms = psms[(psms['Full Sequence'] == psms['Base Sequence']) &
            (psms['Decoy/Contaminant/Target'].isin(['T', 'D'])) &
            (psms['Scan Number'].isin(psm_scans))].copy()
# Fraction of theoretical b/y fragments observed: a peptide of length n has n-1 b ions and n-1 y ions.
psms['fragment_fraction'] = (psms['Matched Ion Counts'].astype(int)
                             / (2 * (psms['Base Sequence'].str.len() - 1)))
mm_targets = psms[psms['Decoy/Contaminant/Target'] == 'T']
mm_decoys  = psms[psms['Decoy/Contaminant/Target'] == 'D']
print(f'{len(mm_targets)} target and {len(mm_decoys)} decoy unmodified PSMs '
      f'(scans matched against ScanNumsWPsms.txt)')

TARGET_COLOR, DECOY_COLOR = '#2a78d6', '#eb6834'   # colorblind-safe blue / orange

# Number line: place each PSM by its MetaMorpheus score; targets above, decoys below.
rng = np.random.default_rng(0)
fig, ax = plt.subplots(figsize=(10, 2.8))
for df, color, y0, label in [(mm_targets, TARGET_COLOR,  0.12, 'Target PSMs'),
                             (mm_decoys,  DECOY_COLOR,  -0.12, 'Decoy PSMs')]:
    x = df['Score'].to_numpy()
    y = y0 + rng.uniform(-0.05, 0.05, size=len(x))   # small vertical jitter so points don't stack
    ax.scatter(x, y, s=45, color=color, alpha=0.7, edgecolor='white',
               linewidth=0.5, label=label, zorder=3)
    med = np.median(x)
    ax.axvline(med, color=color, ls='--', lw=1.5, zorder=2)
    ax.text(med, 0.34, f'median {med:.1f}', color=color, ha='center', va='bottom', fontsize=9)

ax.axhline(0, color='#c3c2b7', lw=1, zorder=1)
ax.set_ylim(-0.42, 0.46)
ax.set_xlabel('MetaMorpheus score')
ax.set_yticks([])
for spine in ['left', 'right', 'top']:
    ax.spines[spine].set_visible(False)
ax.legend(loc='upper right', frameon=False)
ax.set_title('MetaMorpheus score separates target from decoy PSMs (unmodified peptides)')
plt.tight_layout()
plt.show()

One score already pulls the two populations apart, but notice the overlap: some real spectra are noisy and match only a few ions, while some decoys get lucky. Adding a **second** feature, the *fraction* of the peptide's theoretical fragments that were observed, spreads the clouds further apart. (A longer peptide has more fragment ions available to match, so it can accumulate a high raw score without most of its fragments being seen; the fraction controls for that.)

This is the seed of the Percolator idea: no single number cleanly separates good matches from bad ones, but several stacked together carve out a decision boundary.

In [ ]:
# Score vs. fraction of theoretical fragments observed, jittered to expose overlapping PSMs.
jr = np.random.default_rng(1)
fig, ax = plt.subplots(figsize=(7.5, 6))
for df, color, label in [(mm_targets, TARGET_COLOR, 'Target PSMs'),
                         (mm_decoys,  DECOY_COLOR,  'Decoy PSMs')]:
    x = df['Score'].to_numpy()         + jr.uniform(-0.15, 0.15, size=len(df))
    y = df['fragment_fraction'].to_numpy() + jr.uniform(-0.006, 0.006, size=len(df))
    ax.scatter(x, y, s=60, color=color, alpha=0.75, edgecolor='white',
               linewidth=0.6, label=label, zorder=3)

ax.set_xlabel('MetaMorpheus score')
ax.set_ylabel('Fraction of theoretical fragments observed')
ax.set_title('Target vs. decoy PSMs in 2-D feature space')
ax.grid(True, color='#e1e0d9', lw=0.8)
ax.set_axisbelow(True)
for spine in ['right', 'top']:
    ax.spines[spine].set_visible(False)
ax.legend(loc='lower right', frameon=False)
plt.tight_layout()
plt.show()

## 2.2 Database search

A database search engine scores many candidate peptides against every spectrum and reports the best one, along with several scores that describe how well it fits the data.

Let's look at the first 10 PSMs we just loaded, keeping only a few of MetaMorpheus' columns.

In [ ]:
cols = ['Scan Number', 'Base Sequence', 'Precursor Charge', 'Score', 'Delta Score',
        'Mass Diff (Da)', 'Matched Ion Counts', 'Missed Cleavages', 'Decoy/Contaminant/Target', 'QValue']
psms[cols].head(10)

Looking at the table, no single score cleanly divides correct from incorrect PSMs. A high `Score` doesn't guarantee a correct match; a low one doesn't rule it out.

[Percolator (Käll et al., 2007)](https://www.nature.com/articles/nmeth1113) addressed this by collecting multiple scores into a fixed-length vector, then training a [support vector machine (SVM)](https://en.wikipedia.org/wiki/Support_vector_machine) on target and decoy PSMs to find the optimal decision boundary.

## 2.2.1 Why a numerical representation?

Machine learning models are mathematical functions. They cannot accept a raw spectrum, a peptide string like `VLDALDSIK`, or a search-engine report as input. They require a **fixed-length vector of numbers**.

This is the central, simplest form of an encoding problem: how do you convert a PSM, which contains a spectrum, a candidate peptide, and a set of search scores, into a single numerical vector that a model can work with?

Percolator's answer was pragmatic: hand-pick quantities you already trust (match score, mass error, matched ions, charge, etc.), compute them for every PSM, and stack the results into a vector. Every PSM, correct or incorrect, gets compressed to the same dimensions, and the SVM learns which combinations of those numbers distinguish good matches from bad ones.

## 2.3 The Percolator Feature Vector

Percolator's original vector has 20 features. Several of them are specific to the SEQUEST search engine (its preliminary Sp score and rank, the gap to the fifth-best match) or need extra bookkeeping across the whole search (database size per precursor window, protein-level counts). We keep only the features MetaMorpheus reports directly for every PSM, plus the fragment fraction we computed above:

| # | Percolator feature | MetaMorpheus column | Meaning |
|---|--------------------|---------------------|---------|
| 1 | XCorr | `Score` | How well the peptide's theoretical fragments explain the spectrum |
| 2 | $\Delta C_n$ | `Delta Score` | How far the best match beats the runner-up |
| 3 | $\Delta M$ | `Mass Diff (Da)` | Difference between observed and calculated precursor mass |
| 4 | Mass | `Precursor Mass` | Observed (neutral) precursor mass |
| 5 | (none) | `Matched Ion Counts` | Number of b/y fragment ions matched |
| 6 | ionFrac | `fragment_fraction` *(computed)* | Fraction of theoretical b/y fragments observed |
| 7 | enzInt | `Missed Cleavages` | Internal trypsin sites the enzyme skipped |
| 8–10 | charge1–3 | `Precursor Charge` | Charge state, one-hot encoded |

> *Note: MetaMorpheus' versions differ slightly from Percolator's. `Delta Score` is an absolute score difference rather than a fractional one, `Mass Diff (Da)` is observed − calculated (Percolator uses calculated − observed).*

## 2.4 Computing the Features

Let's build the feature vector for our running example: **scan 5672 → VLDALDSIK**. First, isolate its row.

In [ ]:
scan, peptide = 5672, 'VLDALDSIK'
psm = psms.query('`Scan Number` == @scan and `Base Sequence` == @peptide').iloc[0]
psm[['Score', 'Delta Score', 'Mass Diff (Da)', 'Precursor Mass',
     'Matched Ion Counts', 'fragment_fraction', 'Missed Cleavages', 'Precursor Charge']]

### Putting it all together

Most features are copied straight from the row, and the fragment fraction is the column we added earlier. Charge is the exception: it is a *category*, not a quantity (a 3+ precursor isn't "one more" than a 2+ in any meaningful sense), so Percolator **one-hot encodes** it as three booleans. Our precursor is 2+, so we expect `[0, 1, 0]`.

Concatenated, this single Python list **is** our PSM feature vector: the same kind of vector Percolator feeds to its SVM to rerank the first-pass matches. MetaMorpheus does the same thing internally, training a classifier on target and decoy PSMs to compute its posterior error probability (the `PEP` column).

In [ ]:
# MetaMorpheus stores some columns as text (ambiguous PSMs get pipe-joined values),
# so cast each feature to a number.
z = int(psm['Precursor Charge'])

feature_vector = [
    float(psm['Score']),              # 1.   match score       (XCorr)
    float(psm['Delta Score']),        # 2.   best - runner-up  (ΔCn)
    float(psm['Mass Diff (Da)']),     # 3.   mass error        (ΔM)
    float(psm['Precursor Mass']),     # 4.   observed mass     (Mass)
    int(psm['Matched Ion Counts']),   # 5.   matched b/y ions
    float(psm['fragment_fraction']),  # 6.   fragment fraction (ionFrac)
    int(psm['Missed Cleavages']),     # 7.   missed cleavages  (enzInt)
    int(z == 1),                      # 8.   charge 1+
    int(z == 2),                      # 9.   charge 2+
    int(z == 3),                      # 10.  charge 3+
]

assert len(feature_vector) == 10
feature_vector

## 2.5 Limitations and What Comes Next

This short vector works, and Percolator significantly improves identification rates over raw search scores alone, but it comes at a price.

That is to say, these features are **hand-crafted**: they encode exactly the information that experts in 2007 judged to be informative about a PSM. 

I want to get you thinking about signals that didn't make the list.

Peak shape, isotope envelope details, the spatial arrangement of fragment ions, are all discarded. The encoding is also **lossy and order-agnostic**: two completely different spectra that happen to share the same XCorr, mass error, and ion fraction collapse to exactly the same point in vector space.

The remaining notebooks explore what happens when you replace hand-crafted numbers with representations learned from the raw spectrum itself. 

Each further tutorial trades these expert assumptions for learned representations, at increasing computational cost. This PSM feature vector is the baseline. It's quite fast, interpretable, and still the most widely used, but fundamentally limited by what a handful of numbers can see.